# 01 — Local environment setup

This notebook is for a local laptop Jupyter environment. It does not use Google Colab, Google Drive, or network installation. It resolves the local project root, validates the installed environment, creates or loads configuration, and records reproducibility metadata.

Create and populate a local virtual environment from `requirements.txt` before running this notebook.

## 1. Locate the local project

Start Jupyter from the repository root or set `PROJECT_ROOT` before launching Jupyter. The notebook searches the current directory and its parents for `instructions.md`; no user-specific path is embedded in the notebook.

In [1]:
import os
from pathlib import Path

def find_project_root(start_directory):
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / "instructions.md").is_file():
            return candidate.resolve()
    return None

configured_root = os.environ.get("PROJECT_ROOT")
PROJECT_ROOT = (
    Path(configured_root).expanduser().resolve()
    if configured_root else find_project_root(Path.cwd().resolve())
)

if PROJECT_ROOT is None or not (PROJECT_ROOT / "instructions.md").is_file():
    raise FileNotFoundError(
        "Could not find PROJECT_ROOT. Start Jupyter from the repository root or set the PROJECT_ROOT environment variable."
    )

os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Users\shenr\tokenization-research


## 2. Verify the local Python environment

If required packages are missing, this section installs them into the Python environment used by the current Jupyter kernel. On NVIDIA systems it installs the project CUDA 13.0 PyTorch build first, then the remaining packages. Internet access is required only for that initial installation. When installation completes, restart the kernel and rerun the notebook from the beginning.

In [2]:
import importlib.metadata
import platform
import subprocess
import sys
from packaging.version import Version

REQUIRED_PACKAGES = [
    "accelerate", "datasets", "evaluate", "huggingface-hub", "ipywidgets", "pyyaml",
    "scikit-learn", "scipy", "seqeval", "tensorboard", "tokenizers",
    "torch", "transformers",
]
MINIMUM_VERSIONS = {"datasets": "4.8"}

installed_versions = {}
missing_packages = []
for package_name in REQUIRED_PACKAGES:
    try:
        installed_versions[package_name] = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        missing_packages.append(package_name)

outdated_packages = [
    f"{package_name}=={installed_versions[package_name]} (requires >= {minimum_version})"
    for package_name, minimum_version in MINIMUM_VERSIONS.items()
    if package_name in installed_versions and Version(installed_versions[package_name]) < Version(minimum_version)
]

if missing_packages or outdated_packages:
    requirements_path = PROJECT_ROOT / "requirements.txt"
    cuda_requirements_path = PROJECT_ROOT / "requirements-cuda130.txt"
    if not requirements_path.is_file() or not cuda_requirements_path.is_file():
        raise FileNotFoundError("Missing requirements.txt or requirements-cuda130.txt.")
    print(f"Installing or updating packages in: {sys.executable}")
    if missing_packages:
        print(f"Missing packages: {missing_packages}")
    if outdated_packages:
        print(f"Outdated packages: {outdated_packages}")
    if "torch" in missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(cuda_requirements_path)], check=True)
    if any(package_name != "torch" for package_name in missing_packages) or outdated_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)], check=True)
    print("Installation completed. Restart the Jupyter kernel, then rerun this notebook from the beginning.")
else:
    print(f"Python: {sys.version.split()[0]}")
    print(f"Platform: {platform.platform()}")
    for package_name, version in sorted(installed_versions.items()):
        print(f"{package_name}: {version}")

    import torch
    if not torch.cuda.is_available():
        print("WARNING: PyTorch is CPU-only or cannot access the NVIDIA GPU. Install requirements-cuda130.txt, then restart the kernel.")


Python: 3.14.6
Platform: Windows-11-10.0.26200-SP0
accelerate: 1.14.0
datasets: 5.0.0
evaluate: 0.4.6
huggingface-hub: 1.24.0
ipywidgets: 8.1.8
pyyaml: 6.0.3
scikit-learn: 1.9.0
scipy: 1.18.0
seqeval: 1.2.2
tensorboard: 2.21.0
tokenizers: 0.22.2
torch: 2.12.1+cu130
transformers: 5.14.1


## 3. Create or load portable local configuration

The initial configuration uses `project.root: .` and relative paths, so moving the repository on the laptop does not require changing YAML. The file is created only when absent; later runs load it unchanged.

In [3]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CONFIG_PATH.exists():
    default_config = {
        "project": {"root": "."},
        "runtime": {"mode": "train"},
        "checkpointing": {
            "enabled": True, "manifest_path": "experiments/research_pipeline/manifest.json",
        },
        "paths": {
            "raw_data": "data/raw", "processed_data": "data/processed",
            "experiments": "experiments", "outputs": "outputs",
            "tokenizers": "outputs/tokenizers", "datasets": "outputs/datasets",
            "checkpoints": "outputs/checkpoints", "results": "outputs/results",
            "logs": "outputs/logs", "hf_cache": "data/cache/huggingface",
        },
        "data_download": {
            "mode": "capped", "max_text_bytes_per_language": 15_000_000_000, "parquet_shard_rows": 10_000, "languages": ["en", "es", "hi"],
            "sources": {
                "culturax": "uonlp/CulturaX",
            },
            "gluecos": [
                "Huggmachas/GLuecos_NER_EN_HI",
                "Huggmachas/GLuecos_POS_EN_HI_FG",
                "Huggmachas/GLuecos_POS_EN_HI_UD",
                "Huggmachas/GLuecos_POS_EN_ES",
                "Huggmachas/GLuecos_Sentiment_EN_ES",
            ],
        },
        "data_preparation": {
            "manifest_version": 1,
            "preprocessing": {"unicode_normalization": "NFKC", "strip_whitespace": True, "collapse_whitespace": True},
            "filtering": {"min_characters": 1, "max_characters": 1_000_000},
            "splits": {"validation_fraction": 0.005, "hash_algorithm": "sha256"},
            "corpus_storage": {"mode": "manifest_only", "parquet_batch_rows": 10_000},
            "balancing": {"pretokenizer_strategy": "equal_source_text_bytes", "final_strategy": "token_count_after_bpe_training", "source_byte_tolerance": 100_000},
            "downstream": {"preserve_original_columns": True},
        },
        "tokenizer": {
            "vocab_size": 50000, "tokenizer_algorithm": "bpe", "temperature": 1.0,
            "max_candidates": 5, "language_threshold": 0.05, "normalization": "NFKC",
        },
        "probabilistic_tokenizer": {
            "temperature": 1.0, "max_candidates": 5, "min_probability": 0.05,
            "alpha": 1.0, "beta": 1.0,
        },
        "model": {
            "architecture": "XLM-R-base", "layers": 12, "hidden_size": 768,
            "attention_heads": 12, "vocab_size": "configurable", "max_sequence_length": 512,
        },
        "training": {
            "objective": "masked_language_modeling", "mask_probability": 0.15,
            "optimizer": "AdamW", "learning_rate": 1e-4,
            "scheduler": "linear_warmup", "warmup_ratio": 0.06,
            "weight_decay": 0.01, "precision": "fp16_or_bfloat16_if_available",
            "gradient_clipping": 1.0, "batch_size": 32, "epochs": 3,
            "gradient_accumulation_steps": 1, "seed": 42,
        },
        "evaluation": {"seeds": [1, 2, 3, 4, 5], "significance_level": 0.05},
    }
    CONFIG_PATH.write_text(yaml.safe_dump(default_config, sort_keys=False), encoding="utf-8")
    print(f"Created configuration: {CONFIG_PATH}")

with CONFIG_PATH.open(encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file) or {}

# Migrate an older Colab-era configuration without replacing explicit research settings.
def merge_missing(target, defaults):
    changed = False
    for key, value in defaults.items():
        if isinstance(value, dict):
            if key not in target or not isinstance(target[key], dict):
                target[key] = {}
                changed = True
            changed = merge_missing(target[key], value) or changed
        elif key not in target:
            target[key] = value
            changed = True
    return changed

local_defaults = {
    "project": {"root": "."},
    "runtime": {"mode": "train"},
    "checkpointing": {"enabled": True, "manifest_path": "experiments/research_pipeline/manifest.json"},
    "paths": {"hf_cache": "data/cache/huggingface"},
    "data_download": {"mode": "capped", "max_text_bytes_per_language": 15_000_000_000, "parquet_shard_rows": 10_000},
    "data_preparation": {
        "manifest_version": 1,
        "preprocessing": {"unicode_normalization": "NFKC", "strip_whitespace": True, "collapse_whitespace": True},
        "filtering": {"min_characters": 1, "max_characters": 1_000_000},
        "splits": {"validation_fraction": 0.005, "hash_algorithm": "sha256"},
        "corpus_storage": {"mode": "manifest_only", "parquet_batch_rows": 10_000},
        "balancing": {"pretokenizer_strategy": "equal_source_text_bytes", "final_strategy": "token_count_after_bpe_training", "source_byte_tolerance": 100_000},
        "downstream": {"preserve_original_columns": True},
    },
}
configuration_changed = merge_missing(config, local_defaults)
# Migrate an earlier full-corpus download plan to the capped CulturaX-only plan.
download_settings = config.setdefault("data_download", {})
if download_settings.get("mode") == "full":
    download_settings["mode"] = "capped"
    configuration_changed = True
if config.get("project", {}).get("root", "").startswith("/content/"):
    config["project"]["root"] = "."
    configuration_changed = True
if configuration_changed:
    CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    print(f"Migrated local configuration: {CONFIG_PATH}")

print(f"Loaded configuration: {CONFIG_PATH}")
print(f"Runtime mode: {config['runtime']['mode']}")


Loaded configuration: C:\Users\shenr\tokenization-research\configs\config.yaml
Runtime mode: train


## 4. Validate local directories and hardware

This creates only empty configured local artifact directories. It does not download any data, install packages, or begin model training.

In [4]:
import torch

REQUIRED_DIRECTORIES = [
    "src/data", "src/tokenizer", "src/language", "src/models",
    "src/training", "src/evaluation", "configs", "notebooks", "scripts",
]
missing_directories = [name for name in REQUIRED_DIRECTORIES if not (PROJECT_ROOT / name).is_dir()]
if missing_directories:
    raise FileNotFoundError(f"Missing project directories: {missing_directories}")

configured_paths = {name: PROJECT_ROOT / relative for name, relative in config["paths"].items()}
for path in configured_paths.values():
    path.mkdir(parents=True, exist_ok=True)

accelerator_info = {
    "torch": torch.__version__, "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
}
if torch.cuda.is_available():
    accelerator_info["gpu_name"] = torch.cuda.get_device_name(0)
    accelerator_info["gpu_count"] = torch.cuda.device_count()

print("Local repository layout is valid.")
for key, value in accelerator_info.items():
    print(f"{key}: {value}")


Local repository layout is valid.
torch: 2.12.1+cu130
cuda_available: True
cuda_version: 13.0
gpu_name: NVIDIA GeForce RTX 4060 Laptop GPU
gpu_count: 1


## 5. Save local environment metadata

The saved report supports reproducibility and offline diagnostics. It contains no credentials or Hugging Face tokens.

In [5]:
import json
import subprocess
from datetime import datetime, timezone

def command_output(command):
    result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, check=False)
    return result.stdout.strip() if result.returncode == 0 else "unavailable"

environment_report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "config_path": str(CONFIG_PATH),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": installed_versions,
    "accelerator": accelerator_info,
    "git_commit": command_output(["git", "rev-parse", "HEAD"]),
}

setup_directory = configured_paths["experiments"] / "environment_setup"
setup_directory.mkdir(parents=True, exist_ok=True)
environment_path = setup_directory / "environment.json"
environment_path.write_text(json.dumps(environment_report, indent=2), encoding="utf-8")
(setup_directory / "git_commit.txt").write_text(f"{environment_report['git_commit']}\n", encoding="utf-8")

print(f"Saved environment report to: {environment_path}")


Saved environment report to: C:\Users\shenr\tokenization-research\experiments\environment_setup\environment.json


## Setup complete

When internet access is available and `HF_TOKEN` is set for gated datasets, continue with `02_download_datasets.ipynb`. After download completes, the subsequent research workflow is designed to run locally without network access.